In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Given two sorted arrays <code>A</code> of length <code>M</code> and <code>B</code> of length
  <code>N</code>, both containing 32-bit floating-point values in non-decreasing order, produce a
  single sorted array <code>C</code> of length <code>M + N</code> containing all elements of
  <code>A</code> and <code>B</code> in non-decreasing order.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only GPU native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final merged result must be stored in <code>C</code></li>
</ul>

<h2>Example</h2>
<pre>
Input:
  A = [1.0, 3.0, 5.0, 7.0],  M = 4
  B = [2.0, 4.0, 6.0, 8.0],  N = 4

Output:
  C = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]
</pre>

<pre>
Input:
  A = [-1.0, 1.0, 3.0],  M = 3
  B = [2.0],             N = 1

Output:
  C = [-1.0, 1.0, 2.0, 3.0]
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>M</code>, <code>N</code> &le; 50,000,000</li>
  <li><code>M + N</code> &le; 50,000,000</li>
  <li>Both <code>A</code> and <code>B</code> are sorted in non-decreasing order</li>
  <li>Elements are 32-bit floats</li>
  <li>Performance is measured with <code>M</code> = 25,000,000, <code>N</code> = 25,000,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// A, B, C are device pointers (i.e. pointers to memory on the GPU)
extern "C" void solve(const float* A, const float* B, float* C, int M, int N) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# A, B, C are tensors on the GPU
@cute.jit
def solve(A: cute.Tensor, B: cute.Tensor, C: cute.Tensor, M: cute.Uint32, N: cute.Uint32):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# A, B are tensors on GPU
@jax.jit
def solve(A: jax.Array, B: jax.Array, M: int, N: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.memory import UnsafePointer


# A, B, C are device pointers (i.e. pointers to memory on the GPU)
@export
def solve(
    A: UnsafePointer[Float32, MutExternalOrigin],
    B: UnsafePointer[Float32, MutExternalOrigin],
    C: UnsafePointer[Float32, MutExternalOrigin],
    M: Int32,
    N: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# A, B, C are tensors on the GPU
def solve(A: torch.Tensor, B: torch.Tensor, C: torch.Tensor, M: int, N: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# A, B, C are tensors on the GPU
def solve(A: torch.Tensor, B: torch.Tensor, C: torch.Tensor, M: int, N: int):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="Parallel Merge",
            atol=0.0,
            rtol=0.0,
            num_gpus=1,
            access_tier="free",
        )

    def reference_impl(
        self,
        A: torch.Tensor,
        B: torch.Tensor,
        C: torch.Tensor,
        M: int,
        N: int,
    ):
        assert A.shape == (M,), f"Expected A.shape=({M},), got {A.shape}"
        assert B.shape == (N,), f"Expected B.shape=({N},), got {B.shape}"
        assert C.shape == (M + N,), f"Expected C.shape=({M + N},), got {C.shape}"
        assert A.dtype == torch.float32
        assert B.dtype == torch.float32
        assert C.dtype == torch.float32
        assert A.device.type == "cuda"

        merged, _ = torch.sort(torch.cat([A, B]))
        C.copy_(merged)

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "A": (ctypes.POINTER(ctypes.c_float), "in"),
            "B": (ctypes.POINTER(ctypes.c_float), "in"),
            "C": (ctypes.POINTER(ctypes.c_float), "out"),
            "M": (ctypes.c_int, "in"),
            "N": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        A = torch.tensor([1.0, 3.0, 5.0, 7.0], device="cuda", dtype=dtype)
        B = torch.tensor([2.0, 4.0, 6.0, 8.0], device="cuda", dtype=dtype)
        M, N = 4, 4
        C = torch.empty(M + N, device="cuda", dtype=dtype)
        return {"A": A, "B": B, "C": C, "M": M, "N": N}

    def _make_test(self, M: int, N: int, lo: float = -10.0, hi: float = 10.0) -> Dict[str, Any]:
        dtype = torch.float32
        A, _ = torch.sort(torch.empty(M, device="cuda", dtype=dtype).uniform_(lo, hi))
        B, _ = torch.sort(torch.empty(N, device="cuda", dtype=dtype).uniform_(lo, hi))
        C = torch.empty(M + N, device="cuda", dtype=dtype)
        return {"A": A, "B": B, "C": C, "M": M, "N": N}

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.float32
        tests = []

        # Edge cases — tiny sizes
        tests.append(
            {
                "A": torch.tensor([0.0], device="cuda", dtype=dtype),
                "B": torch.tensor([1.0], device="cuda", dtype=dtype),
                "C": torch.empty(2, device="cuda", dtype=dtype),
                "M": 1,
                "N": 1,
            }
        )
        tests.append(
            {
                "A": torch.tensor([2.0], device="cuda", dtype=dtype),
                "B": torch.tensor([-1.0, 1.0, 3.0], device="cuda", dtype=dtype),
                "C": torch.empty(4, device="cuda", dtype=dtype),
                "M": 1,
                "N": 3,
            }
        )
        tests.append(
            {
                "A": torch.tensor([-1.0, 1.0, 3.0], device="cuda", dtype=dtype),
                "B": torch.tensor([2.0], device="cuda", dtype=dtype),
                "C": torch.empty(4, device="cuda", dtype=dtype),
                "M": 3,
                "N": 1,
            }
        )
        # All zeros
        tests.append(
            {
                "A": torch.zeros(2, device="cuda", dtype=dtype),
                "B": torch.zeros(2, device="cuda", dtype=dtype),
                "C": torch.empty(4, device="cuda", dtype=dtype),
                "M": 2,
                "N": 2,
            }
        )

        # Power-of-2 sizes
        tests.append(self._make_test(16, 16))
        tests.append(self._make_test(32, 32, lo=-100.0, hi=0.0))  # all negative
        tests.append(self._make_test(64, 128))
        tests.append(self._make_test(512, 512))
        tests.append(self._make_test(1024, 1024))

        # Non-power-of-2 sizes
        tests.append(self._make_test(30, 33))
        tests.append(self._make_test(100, 77))
        tests.append(self._make_test(255, 127))

        # A entirely less than B (no interleaving needed)
        A_low, _ = torch.sort(torch.empty(256, device="cuda", dtype=dtype).uniform_(-20.0, -10.0))
        B_high, _ = torch.sort(torch.empty(256, device="cuda", dtype=dtype).uniform_(10.0, 20.0))
        tests.append(
            {
                "A": A_low,
                "B": B_high,
                "C": torch.empty(512, device="cuda", dtype=dtype),
                "M": 256,
                "N": 256,
            }
        )

        # Many duplicate values
        A_dup = torch.sort(torch.randint(0, 5, (128,), device="cuda").to(dtype=dtype)).values
        B_dup = torch.sort(torch.randint(0, 5, (128,), device="cuda").to(dtype=dtype)).values
        tests.append(
            {
                "A": A_dup,
                "B": B_dup,
                "C": torch.empty(256, device="cuda", dtype=dtype),
                "M": 128,
                "N": 128,
            }
        )

        # Realistic size
        tests.append(self._make_test(5000, 7000))

        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        M = 25_000_000
        N = 25_000_000
        A, _ = torch.sort(torch.empty(M, device="cuda", dtype=dtype).uniform_(-1.0, 1.0))
        B, _ = torch.sort(torch.empty(N, device="cuda", dtype=dtype).uniform_(-1.0, 1.0))
        C = torch.empty(M + N, device="cuda", dtype=dtype)
        return {"A": A, "B": B, "C": C, "M": M, "N": N}


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
